# Thesis — Final Model (Optuna-tuned, single toggleable dataset)

Train one dual-ResNet50 model on one dataset at a time. Pick the dataset with the `DATASET_DIR` switch in §1, run all cells, and Optuna searches for the best configuration, retrains it at full length, and writes the evaluation artifacts. Run it on Derm7pt on one machine and MILK10k on the other, then compare.

**Why everything is tuned, not fixed:** the strong A6 stack (0.8275 acc) was measured on the *merged* dataset with `mul` fusion. On single datasets the best fusion differs — concat for Derm7pt, add for MILK10k — and MixUp / aux loss / contrastive pretraining hurt on merged. So nothing is hard-coded: every technique is an Optuna on/off toggle and the search decides per dataset.

**Search space:** fusion {mul, add, concat} · lr · dropout · weight_decay · loss {ce, ls, focal}+γ · label_smoothing · use_mixup(+α) · use_aux(+w) · use_swa · use_contrastive. Objective = maximize validation F1-macro; a MedianPruner stops weak trials early; trial 0 is warm-started with the known-good baseline.

## 1 — Setup: imports, dataset toggle, loaders

**Imports, seed, device.** Fixes the random seed (42) for reproducible splits and weights, picks MPS/CUDA/CPU, and quiets Optuna logging. Run once.

In [ ]:
import os, copy, time, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score,
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef, log_loss
)

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
print(f'Device: {device} | PyTorch {torch.__version__} | Optuna {optuna.__version__}')

### Dataset selector

**Pick the dataset here.** Set `DATASET_DIR` to `Path('dataset') / 'Derm7pt'` or
`Path('dataset') / 'Milk10k'`. Everything downstream — split, loaders, search, final
train, eval, all artifacts — runs against this single dataset and is suffixed with
`DATASET_NAME`. To switch: change the line, restart kernel, run all.

**The single switch.** Set `DATASET_DIR` to `Derm7pt` or `Milk10k`. `DATASET_NAME` is derived from it and every split, checkpoint, and output file downstream is suffixed with it — so the two machines never overwrite each other.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                          DATASET SELECTOR                                   ║
# ║   Change this single line to switch between Derm7pt and MILK10k.            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

DATASET_DIR = Path('dataset') / 'Derm7pt'    # or Path('dataset') / 'Derm7pt'

DATASET_NAME = DATASET_DIR.name.lower()
assert DATASET_NAME in ('derm7pt', 'milk10k'), \
    f'Unrecognized DATASET_DIR: {DATASET_DIR}. Expected "Derm7pt" or "Milk10k".'
print(f'>>> DATASET_DIR  = {DATASET_DIR}')
print(f'>>> DATASET_NAME = {DATASET_NAME}')

**Data assembly + loaders.** Builds the dataframe for the chosen dataset (Derm7pt diagnosis→group mapping, or MILK10k metadata+ground-truth pairing of clinical×dermoscopic by `lesion_id`), maps to the fixed 5 classes, defines the paired-image `Dataset` and the train/val transforms, makes the stratified 70/15/15 split with inverse-frequency class weights, and the `DataLoader`s. The train loader uses `drop_last=True` so `BatchNorm1d` / MixUp never see a size-1 batch.

In [ ]:
# ── Dataset-specific dataframe assembly ──────────────────────────────────────
if DATASET_NAME == 'derm7pt':
    derm_raw = pd.read_csv(DATASET_DIR / 'meta' / 'meta.csv')
    derm_raw['clinic_path'] = derm_raw['clinic'].apply(lambda x: str(DATASET_DIR / 'images' / x))
    derm_raw['derm_path']   = derm_raw['derm'].apply(lambda x: str(DATASET_DIR / 'images' / x))
    derm_raw['diagnosis']   = derm_raw['diagnosis'].str.strip().str.lower()
    diagnosis_groups = {
        'melanoma': 'MEL', 'melanoma (less than 0.76 mm)': 'MEL', 'melanoma (in situ)': 'MEL',
        'melanoma (0.76 to 1.5 mm)': 'MEL', 'melanoma (more than 1.5 mm)': 'MEL',
        'melanoma metastasis': 'MEL',
        'clark nevus': 'NV', 'reed or spitz nevus': 'NV', 'dermal nevus': 'NV',
        'blue nevus': 'NV', 'congenital nevus': 'NV', 'combined nevus': 'NV',
        'recurrent nevus': 'NV',
        'basal cell carcinoma': 'BCC',
        'seborrheic keratosis': 'SK',
        'lentigo': 'MISC', 'dermatofibroma': 'MISC', 'vascular lesion': 'MISC',
        'melanosis': 'MISC', 'miscellaneous': 'MISC',
    }
    derm_raw['diagnosis_group'] = derm_raw['diagnosis'].map(diagnosis_groups)
    dataset_df = derm_raw[['clinic_path', 'derm_path', 'diagnosis_group']].copy()
else:  # milk10k
    milk_meta = pd.read_csv(DATASET_DIR / 'MILK10k_Training_Metadata.csv')
    milk_gt   = pd.read_csv(DATASET_DIR / 'MILK10k_Training_GroundTruth.csv')
    milk_class_map = {'MEL': 'MEL', 'NV': 'NV', 'BCC': 'BCC', 'BKL': 'SK',
                      'DF': 'MISC', 'VASC': 'MISC'}
    drop_classes = {'AKIEC', 'SCCKA', 'INF', 'BEN_OTH', 'MAL_OTH'}
    gt_cols = [c for c in milk_gt.columns if c != 'lesion_id']
    milk_gt['raw_class'] = milk_gt[gt_cols].idxmax(axis=1)
    milk_gt = milk_gt[~milk_gt['raw_class'].isin(drop_classes)].copy()
    milk_gt['diagnosis_group'] = milk_gt['raw_class'].map(milk_class_map)
    clinic_meta = (milk_meta[milk_meta['image_type'] == 'clinical: close-up']
                   [['lesion_id', 'isic_id']].rename(columns={'isic_id': 'clinic_isic'}))
    derm_meta   = (milk_meta[milk_meta['image_type'] == 'dermoscopic']
                   [['lesion_id', 'isic_id']].rename(columns={'isic_id': 'derm_isic'}))
    paths = clinic_meta.merge(derm_meta, on='lesion_id')
    milk_raw = milk_gt[['lesion_id', 'diagnosis_group']].merge(paths, on='lesion_id')
    milk_raw['clinic_path'] = milk_raw.apply(
        lambda r: str(DATASET_DIR / 'MILK10k_Training_Input' / r['lesion_id'] / f"{r['clinic_isic']}.jpg"), axis=1)
    milk_raw['derm_path'] = milk_raw.apply(
        lambda r: str(DATASET_DIR / 'MILK10k_Training_Input' / r['lesion_id'] / f"{r['derm_isic']}.jpg"), axis=1)
    dataset_df = milk_raw[['clinic_path', 'derm_path', 'diagnosis_group']].copy()

# 5-class label space (fixed across both datasets so artifacts remain comparable)
class_names = ['BCC', 'MEL', 'MISC', 'NV', 'SK']
label_map   = {name: i for i, name in enumerate(class_names)}
dataset_df['label'] = dataset_df['diagnosis_group'].map(label_map)
dataset_df = dataset_df.dropna(subset=['label']).reset_index(drop=True)
dataset_df['label'] = dataset_df['label'].astype(int)
print(f'{DATASET_NAME}: {len(dataset_df)} samples | {len(class_names)} classes ({class_names})')
print(dataset_df['diagnosis_group'].value_counts().to_string())


class SkinLesionDualDataset(Dataset):
    """Returns (clinic_img, derm_img, label) for each sample."""
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clinic_img = Image.open(row['clinic_path']).convert('RGB')
        derm_img   = Image.open(row['derm_path']).convert('RGB')
        if self.transform:
            clinic_img = self.transform(clinic_img)
            derm_img   = self.transform(derm_img)
        return clinic_img, derm_img, torch.tensor(row['label'], dtype=torch.long)


train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ── Stratified 70/15/15 split + class weights ────────────────────────────────
def build_dataset(df_in, name):
    labels  = df_in['label'].values.astype(int)
    indices = np.arange(len(df_in))
    tr, tmp = train_test_split(indices, test_size=0.30, stratify=labels, random_state=SEED)
    va, te  = train_test_split(tmp,     test_size=0.50, stratify=labels[tmp], random_state=SEED)
    train_labels = labels[tr]
    class_counts = np.bincount(train_labels, minlength=len(class_names))
    safe_counts  = np.where(class_counts == 0, 1, class_counts)
    sample_w     = (1.0 / safe_counts)[train_labels]
    loss_w       = torch.tensor(
        [len(train_labels) / (len(class_names) * c) if c > 0 else 0.0 for c in class_counts],
        dtype=torch.float32).to(device)
    return {
        'name': name, 'df': df_in,
        'train_idx': tr, 'val_idx': va, 'test_idx': te,
        'sample_weights': sample_w, 'loss_weights': loss_w,
        'class_counts': class_counts,
    }

ACTIVE_DS = build_dataset(dataset_df, DATASET_NAME)
print(f'{ACTIVE_DS["name"]:>8s}: train={len(ACTIVE_DS["train_idx"])} '
      f'val={len(ACTIVE_DS["val_idx"])} test={len(ACTIVE_DS["test_idx"])} '
      f'| class counts (train) = {ACTIVE_DS["class_counts"].tolist()}')

# ── DataLoaders (train uses drop_last=True so BatchNorm1d / MixUp never see size-1) ──
# Windows + CUDA now get parallel data workers (was force-disabled on nt, which
# serialized image decode/aug on the main thread and starved the GPU). MPS/CPU keep 0.
_NUM_WORKERS = 0 if device.type in ('mps', 'cpu') else min(8, os.cpu_count() or 4)
_PIN_MEMORY  = device.type == 'cuda'

def make_loaders(batch_size, ds):
    """Return (train_loader, val_loader, test_loader, test_ds) for a dataset dict."""
    train_ds = SkinLesionDualDataset(ds['df'].iloc[ds['train_idx']], train_transform)
    val_ds   = SkinLesionDualDataset(ds['df'].iloc[ds['val_idx']],   val_transform)
    test_ds  = SkinLesionDualDataset(ds['df'].iloc[ds['test_idx']],  val_transform)
    samp = WeightedRandomSampler(ds['sample_weights'], len(ds['sample_weights']), replacement=True)
    kw = dict(num_workers=_NUM_WORKERS, pin_memory=_PIN_MEMORY,
              persistent_workers=(_NUM_WORKERS > 0))
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=samp, drop_last=True, **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)
    return train_loader, val_loader, test_loader, test_ds

## 2 — Model + technique components

Backbone, focal loss, paired MixUp, the flexible dual-branch model (now with a
**concat** fusion mode added alongside add/mul), NT-Xent contrastive loss, the
train/validate loops, SWA BN-recompute, early stopping, and contrastive pretraining.
All copied from `thesis_improved.ipynb` (proven) and extended.

**Backbone.** ResNet50 with the classifier head stripped, returning 2048×7×7 spatial feature maps. One instance is created per modality (clinic, derm).

In [ ]:
class ResNet50Backbone(nn.Module):
    """ResNet50 feature extractor — returns spatial feature maps [B, 2048, 7, 7]."""
    def __init__(self, pretrained=True):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4)

    def forward(self, x):
        return self.features(x)

**Loss + augmentation building blocks.** `FocalLoss` handles hard *or* soft targets with per-class weights and label smoothing (γ=0 reduces to weighted/smoothed cross-entropy). `mixup_dual_batch` applies the *same* blend factor to both modalities so the lesion pairing survives, and returns a soft target.

In [ ]:
class FocalLoss(nn.Module):
    """Focal loss accepting HARD [B] or SOFT [B,C] targets, with optional label
    smoothing and per-class alpha. gamma=0 reduces to (weighted, smoothed) CE."""
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha               # tensor [C] or None
        self.ls = label_smoothing
        self.reduction = reduction

    def forward(self, logits, target):
        C = logits.size(1)
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        if target.dim() == 1:
            q = F.one_hot(target, C).float()
        else:
            q = target.float()
        if self.ls > 0:
            q = q * (1 - self.ls) + self.ls / C
        loss_terms = -((1 - p) ** self.gamma) * q * logp        # [B, C]
        if self.alpha is not None:
            loss_terms = loss_terms * self.alpha.view(1, -1)
        loss = loss_terms.sum(dim=1)                             # [B]
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


def mixup_dual_batch(clinic, derm, labels, num_classes, alpha):
    """Paired MixUp: SAME lambda + permutation applied to both modalities so the
    (clinic, derm) lesion pairing is preserved. Returns mixed images + SOFT [B,C] target."""
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(clinic.size(0), device=clinic.device)
    clinic_m = lam * clinic + (1 - lam) * clinic[perm]
    derm_m   = lam * derm   + (1 - lam) * derm[perm]
    y = F.one_hot(labels, num_classes).float()
    target = lam * y + (1 - lam) * y[perm]
    return clinic_m, derm_m, target

**The model.** Two ResNet50 backbones fused by `add` / `mul` / `concat` (concat uses a 1×1 conv to fold the doubled 4096 channels back to 2048). Optional aux head on the derm branch and projection heads for contrastive pretraining. `forward()` returns `(main_logits, aux_logits)`; the freeze/unfreeze helpers drive the staged fine-tuning.

In [ ]:
class ImprovedDualBranchFusion(nn.Module):
    """Flexible dual-branch model. fusion in {add, mul, concat}.

    forward() -> (main_logits, aux_logits)   # aux None unless use_aux=True
    encode()  -> (pooled_clinic, pooled_derm) [B, 2048]
    project() -> (z_clinic, z_derm) L2-normalized [B, proj_dim]   # NT-Xent
    """
    def __init__(self, num_classes=5, dropout=0.32, fusion='mul', pretrained=True,
                 use_aux=False, use_projection=False, proj_dim=128):
        super().__init__()
        assert fusion in ('add', 'mul', 'concat')
        self.fusion = fusion
        self.use_aux = use_aux
        self.use_projection = use_projection
        self.resnet_clinic = ResNet50Backbone(pretrained=pretrained)
        self.resnet_derm   = ResNet50Backbone(pretrained=pretrained)
        # concat doubles channels (2048+2048) -> 1x1 conv projects back to 2048
        if fusion == 'concat':
            self.fuse_conv = nn.Conv2d(4096, 2048, kernel_size=1, bias=False)
        self.post_fuse = nn.Sequential(nn.BatchNorm2d(2048), nn.ReLU(inplace=True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(512, num_classes))
        if use_aux:
            self.aux_head = nn.Sequential(
                nn.Linear(2048, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
                nn.Dropout(dropout), nn.Linear(256, num_classes))
        if use_projection:
            self.proj_clinic = nn.Sequential(
                nn.Linear(2048, 2048), nn.ReLU(inplace=True), nn.Linear(2048, proj_dim))
            self.proj_derm = nn.Sequential(
                nn.Linear(2048, 2048), nn.ReLU(inplace=True), nn.Linear(2048, proj_dim))

    def _freeze_backbones(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.parameters(): p.requires_grad = False
            for p in m.features[7].parameters(): p.requires_grad = True

    def unfreeze_resnets(self):
        for m in [self.resnet_clinic, self.resnet_derm]:
            for p in m.features[6].parameters(): p.requires_grad = True

    def encode(self, clinic_img, derm_img):
        fc = self.pool(self.resnet_clinic(clinic_img)).flatten(1)
        fd = self.pool(self.resnet_derm(derm_img)).flatten(1)
        return fc, fd

    def project(self, clinic_img, derm_img):
        fc, fd = self.encode(clinic_img, derm_img)
        return F.normalize(self.proj_clinic(fc), dim=1), F.normalize(self.proj_derm(fd), dim=1)

    def forward(self, clinic_img, derm_img):
        feat_c = self.resnet_clinic(clinic_img)
        feat_d = self.resnet_derm(derm_img)
        if self.fusion == 'add':
            fused = feat_c + feat_d
        elif self.fusion == 'mul':
            fused = feat_c * feat_d
        else:  # concat
            fused = self.fuse_conv(torch.cat([feat_c, feat_d], dim=1))
        x = self.pool(self.post_fuse(fused)).flatten(1)
        main = self.classifier(x)
        aux = self.aux_head(self.pool(feat_d).flatten(1)) if self.use_aux else None
        return main, aux

**Contrastive pretraining (optional technique).** Cross-modal SimCLR: NT-Xent treats `(clinic, derm)` of the same lesion as the positive pair against all other in-batch samples. `contrastive_pretrain` warms both backbones this way and saves their weights for reuse when a trial enables `use_contrastive`.

In [ ]:
def nt_xent_loss(z_clinic, z_derm, temperature=0.5):
    """Cross-modal NT-Xent (SimCLR). Positive pair = (clinic_i, derm_i) of the same
    lesion; all other 2B-2 samples in the batch are negatives."""
    B = z_clinic.size(0)
    z = torch.cat([z_clinic, z_derm], dim=0)              # [2B, D]
    sim = torch.mm(z, z.t()) / temperature                # [2B, 2B]
    sim.masked_fill_(torch.eye(2 * B, dtype=torch.bool, device=z.device), float('-inf'))
    targets = (torch.arange(2 * B, device=z.device) + B) % (2 * B)
    return F.cross_entropy(sim, targets)


def contrastive_pretrain(ds, ckpt_out, epochs, base_lr, weight_decay, temp, grad_clip, batch_size):
    """Cross-modal SimCLR pretraining of the two backbones. Saves backbone state dicts."""
    train_loader, *_ = make_loaders(batch_size, ds)
    model = ImprovedDualBranchFusion(num_classes=len(class_names), dropout=DROPOUT,
                                     fusion='mul', pretrained=True, use_projection=True).to(device)
    for p in model.parameters():
        p.requires_grad = True
    params = (list(model.resnet_clinic.parameters()) + list(model.resnet_derm.parameters())
              + list(model.proj_clinic.parameters()) + list(model.proj_derm.parameters()))
    opt = optim.AdamW(params, lr=base_lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    start = time.time()
    for epoch in range(epochs):
        model.train()
        tot, n = 0.0, 0
        for clinic, derm, _ in train_loader:
            clinic, derm = clinic.to(device), derm.to(device)
            opt.zero_grad()
            zc, zd = model.project(clinic, derm)
            loss = nt_xent_loss(zc, zd, temp)
            loss.backward()
            nn.utils.clip_grad_norm_(params, grad_clip)
            opt.step()
            tot += loss.item() * clinic.size(0); n += clinic.size(0)
        sched.step()
        print(f'  [contrastive] ep {epoch+1:2d}/{epochs} | loss {tot/n:.4f}')
    torch.save({'resnet_clinic': model.resnet_clinic.state_dict(),
                'resnet_derm':   model.resnet_derm.state_dict()}, ckpt_out)
    print(f'  Saved contrastive backbones -> {ckpt_out} ({(time.time()-start)/60:.1f} min)')
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return ckpt_out

**Training plumbing.** Warmup→cosine LR schedule, the one-epoch train loop (MixUp + aux aware), the validation loop, `val_f1_macro` (the Optuna objective and pruning signal), `update_bn_dual` (recomputes BatchNorm stats for the dual-input SWA model), and `EarlyStopping`.

In [ ]:
def warmup_cosine_lambda(warmup_epochs, phase_epochs):
    """Linear warmup for `warmup_epochs`, then cosine decay over the rest of the phase."""
    def lr_lambda(ep):
        if warmup_epochs and ep < warmup_epochs:
            return (ep + 1) / warmup_epochs
        prog = (ep - warmup_epochs) / max(1, phase_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return lr_lambda


def train_one_epoch_improved(model, loader, optimizer, criterion, grad_clip,
                             num_classes, mixup_alpha=0.0, mixup_prob=0.0, aux_weight=0.0,
                             scaler=None):
    model.train()
    use_amp = scaler is not None and scaler.is_enabled()
    total_loss, correct, total = 0.0, 0, 0
    for clinic, derm, labels in loader:
        clinic, derm, labels = clinic.to(device), derm.to(device), labels.to(device)
        optimizer.zero_grad()
        if mixup_prob > 0 and np.random.rand() < mixup_prob:
            clinic, derm, target = mixup_dual_batch(clinic, derm, labels, num_classes, mixup_alpha)
        else:
            target = labels
        with torch.autocast(device_type='cuda', enabled=use_amp):
            main_logits, aux_logits = model(clinic, derm)
            loss = criterion(main_logits, target)
            if aux_weight > 0 and aux_logits is not None:
                loss = loss + aux_weight * criterion(aux_logits, target)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        total_loss += loss.item() * clinic.size(0)
        hard = target.argmax(1) if target.dim() == 2 else target
        correct += main_logits.argmax(1).eq(hard).sum().item()
        total   += clinic.size(0)
    return total_loss / total, correct / total


def validate_improved(model, loader, criterion):
    """Validation always uses hard labels, no MixUp, no aux term."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for clinic, derm, labels in loader:
            clinic, derm, labels = clinic.to(device), derm.to(device), labels.to(device)
            main_logits, _ = model(clinic, derm)
            loss = criterion(main_logits, labels)
            total_loss += loss.item() * clinic.size(0)
            correct += main_logits.argmax(1).eq(labels).sum().item()
            total   += clinic.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def val_f1_macro(model, loader):
    """Val F1-macro — the Optuna objective / pruning signal. No MixUp, hard labels."""
    model.eval()
    preds, y = [], []
    for clinic, derm, labels in loader:
        main_logits, _ = model(clinic.to(device), derm.to(device))
        preds.extend(main_logits.argmax(1).cpu().numpy())
        y.extend(labels.numpy())
    return f1_score(np.array(y), np.array(preds), average='macro', zero_division=0)


def update_bn_dual(loader, model):
    """Recompute BatchNorm running stats for a dual-input model (torch's update_bn
    assumes a single input tensor)."""
    momenta = {}
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.reset_running_stats()
            momenta[module] = module.momentum
            module.momentum = None
    if not momenta:
        return
    was_training = model.training
    model.train()
    with torch.no_grad():
        for clinic, derm, _ in loader:
            model(clinic.to(device), derm.to(device))
    for bn, m in momenta.items():
        bn.momentum = m
    model.train(was_training)


class EarlyStopping:
    """Stops training if val_loss doesn't improve for `patience` consecutive epochs."""
    def __init__(self, patience=20, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.counter = 0; self.best_loss = float('inf')
        self.best_state = None; self.stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

**Test-set evaluator.** Runs a finished model over the test loader and returns the 7 thesis metrics (accuracy, balanced accuracy, F1-macro/weighted, Cohen κ, MCC, log-loss) plus raw labels/preds/probs for the confusion matrix.

In [ ]:
@torch.no_grad()
def evaluate_model_full(model, test_loader):
    """Run a built model over the test set; return (metrics dict, y, preds, probs)."""
    model.eval()
    preds, y, probs = [], [], []
    for clinic, derm, lb in test_loader:
        logits, _ = model(clinic.to(device), derm.to(device))
        p = F.softmax(logits, dim=1)
        preds.extend(logits.argmax(1).cpu().numpy())
        y.extend(lb.numpy()); probs.extend(p.cpu().numpy())
    y = np.array(y); preds = np.array(preds); probs = np.array(probs)
    nc = len(class_names)
    metrics = {
        'accuracy':          accuracy_score(y, preds),
        'balanced_accuracy': balanced_accuracy_score(y, preds),
        'f1_macro':          f1_score(y, preds, average='macro', zero_division=0),
        'f1_weighted':       f1_score(y, preds, average='weighted', zero_division=0),
        'cohen_kappa':       cohen_kappa_score(y, preds),
        'mcc':               matthews_corrcoef(y, preds),
        'log_loss':          log_loss(y, probs, labels=list(range(nc))),
    }
    return metrics, y, preds, probs

### Constants, search/final scale, artifact paths

`SMOKE_TEST=True` validates the whole chain (toggle → loaders → train_trial → study →
final train → eval → PNGs) in minutes. Flip to `False` for the real search.

**Run-scale config + artifact paths.** Structural constants (batch size, unfreeze epoch, etc.) and the `SMOKE_TEST` switch: `True` runs a tiny 2-trial × 2-epoch dry run to validate the whole chain in minutes; set `False` for the real 35-trial search. Also defines every per-dataset output path (study DB, checkpoint, JSON/CSV, PNGs).

In [ ]:
# ── Fixed structural constants (match prior thesis runs) ─────────────────────
DROPOUT        = 0.32          # default for contrastive pretrain + fallback
BATCH_SIZE     = 64
UNFREEZE_EPOCH = 5
GRAD_CLIP      = 1.0
USE_AMP        = (device.type == 'cuda')   # mixed precision — CUDA only (no-op on MPS/CPU)
WARMUP_EPOCHS  = 3
SWA_START_FRAC = 0.75
MIXUP_PROB     = 0.5
PROJ_DIM       = 128
CONTRASTIVE_TEMP   = 0.5
CONTRASTIVE_EPOCHS = 30

# ── Run-scale config ─────────────────────────────────────────────────────────
SMOKE_TEST = True     # True → tiny/fast validation; False → full search

if SMOKE_TEST:
    N_TRIALS        = 2
    SEARCH_EPOCHS   = 2
    SEARCH_PATIENCE = 99
    FINAL_EPOCHS    = 3
    FINAL_PATIENCE  = 99
    CONTRASTIVE_EPOCHS = 2
else:
    N_TRIALS        = 35
    SEARCH_EPOCHS   = 28
    SEARCH_PATIENCE = 10
    FINAL_EPOCHS    = 60
    FINAL_PATIENCE  = 20

OBJECTIVE_METRIC = 'val_f1_macro'   # maximize

# ── Per-dataset artifact paths (suffixed so two machines never clash) ────────
SUFFIX          = '_smoke' if SMOKE_TEST else ''
STUDY_DB        = f'sqlite:///optuna_{DATASET_NAME}{SUFFIX}.db'
STUDY_NAME      = f'thesis_final_{DATASET_NAME}{SUFFIX}'
BEST_CKPT       = f'thesis_final_{DATASET_NAME}{SUFFIX}_best.pth'
BEST_PARAMS_JSON= f'thesis_final_{DATASET_NAME}{SUFFIX}_bestparams.json'
CONTRASTIVE_CKPT= f'contrastive_{DATASET_NAME}{SUFFIX}.pth'
RESULTS_CSV     = f'results_{DATASET_NAME}{SUFFIX}.csv'
RESULTS_JSON    = f'results_{DATASET_NAME}{SUFFIX}.json'
CONF_PNG        = f'confusion_{DATASET_NAME}{SUFFIX}.png'
CURVES_PNG      = f'curves_{DATASET_NAME}{SUFFIX}.png'

print(f'Dataset      : {DATASET_NAME}  (SMOKE_TEST={SMOKE_TEST})')
print(f'Search       : {N_TRIALS} trials x up to {SEARCH_EPOCHS} epochs (patience {SEARCH_PATIENCE})')
print(f'Final retrain: {FINAL_EPOCHS} epochs (patience {FINAL_PATIENCE})')
print(f'Study storage: {STUDY_DB}')
print(f'Best ckpt    : {BEST_CKPT}')

## 3 — Optuna search

`train_trial` is a self-contained trainer that takes **every** hyperparameter as an
explicit argument (the `thesis_improved.train_model` driver read them from globals, so
it could not be tuned). It builds the loss + optimizer inline, runs the
freeze→unfreeze@5 + warmup→cosine schedule, optional SWA, and reports per-epoch val
F1-macro to Optuna for pruning. During the SWA averaging phase the *running* model's
F1 is reported (the averaged model has stale BatchNorm until `update_bn_dual`); the
returned objective uses the BN-corrected SWA model.

**Core trainer (search + final).** Takes every hyperparameter as an explicit argument (so Optuna can vary them — the old `train_model` read them from globals and couldn't be tuned), builds loss + optimizer inline, runs freeze→unfreeze@5 + warmup→cosine + optional SWA, reports per-epoch val F1 to Optuna so weak trials get pruned, and returns the best/final F1. During SWA it reports the *running* model's F1 (stable BatchNorm) and BN-corrects the averaged model once at the end.

In [ ]:
def train_trial(*, fusion, lr, dropout, weight_decay, loss_type, label_smoothing,
                focal_gamma, use_mixup, mixup_alpha, use_aux, aux_weight, use_swa,
                ds=None, total_epochs=None, patience=None, schedule='warmup_cosine',
                contrastive_ckpt=None, ckpt_out=None, trial=None, verbose=False):
    """Train one configuration. Returns (best/final val_f1_macro, history, state_dict).
    Used for BOTH Optuna trials (trial set) and the final retrain (trial=None)."""
    ds           = ACTIVE_DS if ds is None else ds
    total_epochs = SEARCH_EPOCHS if total_epochs is None else total_epochs
    patience     = SEARCH_PATIENCE if patience is None else patience
    NC = len(class_names)
    train_loader, val_loader, _, _ = make_loaders(BATCH_SIZE, ds)

    model = ImprovedDualBranchFusion(num_classes=NC, dropout=dropout, fusion=fusion,
                                     pretrained=True, use_aux=use_aux).to(device)
    if contrastive_ckpt and Path(contrastive_ckpt).exists():
        bb = torch.load(contrastive_ckpt, map_location=device, weights_only=False)
        model.resnet_clinic.load_state_dict(bb['resnet_clinic'])
        model.resnet_derm.load_state_dict(bb['resnet_derm'])
        if verbose: print(f'  loaded contrastive backbones from {contrastive_ckpt}')
    model._freeze_backbones()

    # ── loss (inline so focal_gamma / label_smoothing come from args) ──
    lw = ds['loss_weights']
    if loss_type == 'ce':
        criterion = nn.CrossEntropyLoss(weight=lw)
    elif loss_type == 'ls':
        criterion = nn.CrossEntropyLoss(weight=lw, label_smoothing=label_smoothing)
    else:  # focal
        criterion = FocalLoss(gamma=focal_gamma, alpha=lw, label_smoothing=label_smoothing)

    # ── optimizer + scheduler (inline so lr / weight_decay come from args) ──
    def build_os(params, base_lr, phase_ep, warmup):
        opt = optim.AdamW(list(params), lr=base_lr, weight_decay=weight_decay)
        if warmup and warmup > 0:
            sched = optim.lr_scheduler.LambdaLR(opt, warmup_cosine_lambda(warmup, phase_ep))
        else:
            sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, phase_ep))
        return opt, sched

    warmup = WARMUP_EPOCHS if schedule == 'warmup_cosine' else 0
    opt, sched = build_os(filter(lambda p: p.requires_grad, model.parameters()),
                          lr, UNFREEZE_EPOCH, warmup)

    swa_model = AveragedModel(model) if use_swa else None
    swa_start = int(SWA_START_FRAC * total_epochs)
    swa_sched = None
    mixup_alpha_ = mixup_alpha if use_mixup else 0.0
    mixup_prob_  = MIXUP_PROB  if use_mixup else 0.0
    aux_weight_  = aux_weight  if use_aux   else 0.0

    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    early = EarlyStopping(patience=patience)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
    best_f1 = 0.0
    start = time.time()

    for epoch in range(total_epochs):
        if epoch == UNFREEZE_EPOCH:
            model.unfreeze_resnets()
            opt, sched = build_os(model.parameters(), lr / 4, total_epochs - UNFREEZE_EPOCH, 0)
            if use_swa:
                swa_sched = SWALR(opt, swa_lr=lr / 8)

        tr_loss, tr_acc = train_one_epoch_improved(
            model, train_loader, opt, criterion, GRAD_CLIP, NC,
            mixup_alpha_, mixup_prob_, aux_weight_, scaler=scaler)
        va_loss, va_acc = validate_improved(model, val_loader, criterion)

        in_swa = use_swa and epoch >= swa_start
        if in_swa:
            swa_model.update_parameters(model)
            if swa_sched is not None: swa_sched.step()
        else:
            sched.step()

        # pruning / tracking signal uses the RUNNING model (stable BN)
        f1 = val_f1_macro(model, val_loader)
        history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)
        history['val_f1'].append(f1)
        best_f1 = max(best_f1, f1)

        if not use_swa:
            early(va_loss, model)
        if verbose:
            tag = 'SWA' if in_swa else f'es {early.counter}/{patience}'
            print(f'  ep {epoch+1:2d}/{total_epochs} | train {tr_loss:.4f}/{tr_acc:.4f} | '
                  f'val {va_loss:.4f}/{va_acc:.4f} | f1 {f1:.4f} | {tag}')

        if trial is not None:
            trial.report(f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if (not use_swa) and early.stop:
            early.restore(model)
            break

    # ── finalize ──
    if use_swa:
        update_bn_dual(train_loader, swa_model)
        final_model = swa_model.module
        final_f1 = val_f1_macro(final_model, val_loader)
    else:
        early.restore(model)
        final_model = model
        final_f1 = max(val_f1_macro(final_model, val_loader), best_f1)

    state = copy.deepcopy(final_model.state_dict())
    if ckpt_out:
        torch.save({'model_state_dict': state, 'fusion': fusion, 'dropout': dropout,
                    'use_aux': use_aux, 'class_names': class_names, 'label_map': label_map,
                    'history': history, 'val_f1_macro': final_f1,
                    'config': {'fusion': fusion, 'lr': lr, 'dropout': dropout,
                               'weight_decay': weight_decay, 'loss_type': loss_type,
                               'label_smoothing': label_smoothing, 'focal_gamma': focal_gamma,
                               'use_mixup': use_mixup, 'mixup_alpha': mixup_alpha,
                               'use_aux': use_aux, 'aux_weight': aux_weight,
                               'use_swa': use_swa}}, ckpt_out)
    if verbose:
        print(f'  done in {(time.time()-start)/60:.1f} min | val f1_macro {final_f1:.4f}')
    del model
    if use_swa: del swa_model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return final_f1, history, state

**Optuna search space.** Samples fusion, lr, dropout, weight_decay, loss+γ, label smoothing, and the on/off toggles (mixup/aux/SWA/contrastive), then calls `train_trial`. Conditional params (e.g. `mixup_alpha`) are only sampled when their toggle is on, so the search space stays tight.

In [ ]:
def objective(trial):
    fusion  = trial.suggest_categorical('fusion', ['mul', 'add', 'concat'])
    lr      = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.2, 0.5)
    wd      = trial.suggest_float('weight_decay', 1e-5, 3e-3, log=True)
    loss_type = trial.suggest_categorical('loss', ['ce', 'ls', 'focal'])
    ls    = trial.suggest_float('label_smoothing', 0.0, 0.15) if loss_type in ('ls', 'focal') else 0.0
    gamma = trial.suggest_float('focal_gamma', 1.0, 3.0) if loss_type == 'focal' else 2.0
    use_mixup = trial.suggest_categorical('use_mixup', [True, False])
    mixup_a   = trial.suggest_float('mixup_alpha', 0.1, 0.5) if use_mixup else 0.0
    use_aux = trial.suggest_categorical('use_aux', [True, False])
    aux_w   = trial.suggest_float('aux_weight', 0.1, 0.5) if use_aux else 0.0
    use_swa = trial.suggest_categorical('use_swa', [True, False])
    use_contrastive = trial.suggest_categorical('use_contrastive', [True, False])

    cc = None
    if use_contrastive:
        cc = CONTRASTIVE_CKPT
        if not Path(cc).exists():
            print(f'  [trial {trial.number}] building contrastive backbones (one-time)...')
            contrastive_pretrain(ACTIVE_DS, cc, CONTRASTIVE_EPOCHS, lr, wd,
                                 CONTRASTIVE_TEMP, GRAD_CLIP, BATCH_SIZE)

    f1, _, _ = train_trial(
        fusion=fusion, lr=lr, dropout=dropout, weight_decay=wd, loss_type=loss_type,
        label_smoothing=ls, focal_gamma=gamma, use_mixup=use_mixup, mixup_alpha=mixup_a,
        use_aux=use_aux, aux_weight=aux_w, use_swa=use_swa,
        ds=ACTIVE_DS, total_epochs=SEARCH_EPOCHS, patience=SEARCH_PATIENCE,
        contrastive_ckpt=cc, ckpt_out=None, trial=trial)
    return f1

**Run the search.** Creates (or resumes) the per-dataset SQLite study, warm-starts trial 0 with the known-good baseline (concat for Derm7pt, add for MILK10k), then optimizes. SQLite storage + `load_if_exists` makes it resumable across kernel restarts.

In [ ]:
# Known-good baseline to warm-start trial 0 (fusion differs by dataset).
_baseline_fusion = 'concat' if DATASET_NAME == 'derm7pt' else 'add'
WARM_START = {
    'fusion': _baseline_fusion, 'lr': 0.00075, 'dropout': 0.32, 'weight_decay': 0.000957,
    'loss': 'ls', 'label_smoothing': 0.1, 'use_mixup': False, 'use_aux': False,
    'use_swa': True, 'use_contrastive': False,
}

study = optuna.create_study(
    study_name=STUDY_NAME, storage=STUDY_DB, direction='maximize',
    sampler=TPESampler(seed=SEED, multivariate=True),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=8, interval_steps=1),
    load_if_exists=True)

if len(study.trials) == 0:
    study.enqueue_trial(WARM_START)
    print(f'Enqueued warm-start baseline (fusion={_baseline_fusion}).')

remaining = max(0, N_TRIALS - len(study.trials))
print(f'Study "{STUDY_NAME}" has {len(study.trials)} trials; running {remaining} more...')
if remaining > 0:
    study.optimize(objective, n_trials=remaining, gc_after_trial=True)

print(f'\nBest val f1_macro = {study.best_value:.4f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k:<16}: {v}')

**Persist the winner.** Writes the best params + value to JSON and, if plotting deps are present, saves the Optuna optimization-history and parameter-importance figures.

In [ ]:
best_record = {
    'dataset': DATASET_NAME, 'objective': OBJECTIVE_METRIC,
    'best_value': study.best_value, 'best_trial': study.best_trial.number,
    'best_params': study.best_params, 'n_trials': len(study.trials),
}
with open(BEST_PARAMS_JSON, 'w') as f:
    json.dump(best_record, f, indent=2)
print(f'Saved best params -> {BEST_PARAMS_JSON}')

# Optional Optuna plots (guarded — plotly/kaleido may be absent)
try:
    from optuna.visualization.matplotlib import (plot_optimization_history,
                                                  plot_param_importances)
    fig1 = plot_optimization_history(study); plt.tight_layout()
    plt.savefig(f'optuna_history_{DATASET_NAME}{SUFFIX}.png', dpi=150, bbox_inches='tight'); plt.show()
    if len([t for t in study.trials if t.state.name == 'COMPLETE']) >= 2:
        fig2 = plot_param_importances(study); plt.tight_layout()
        plt.savefig(f'optuna_importance_{DATASET_NAME}{SUFFIX}.png', dpi=150, bbox_inches='tight'); plt.show()
except Exception as e:
    print(f'(skipped Optuna plots: {e})')

## 4 — Final full train on best params

Rebuild the winning configuration and train it at full length (`FINAL_EPOCHS`). Uses the
identical `train_trial` so the final model matches the searched config exactly, just
longer. If the best config used contrastive init, the backbones are pretrained first.

**Final full train.** Rebuilds the winning configuration and trains it at full length (`FINAL_EPOCHS`) via the same `train_trial`, pretraining the backbones first if the best config used contrastive init. Saves the final checkpoint.

In [ ]:
bp = study.best_params
b_fusion  = bp['fusion']
b_lr      = bp['lr']
b_dropout = bp['dropout']
b_wd      = bp['weight_decay']
b_loss    = bp['loss']
b_ls      = bp.get('label_smoothing', 0.0)
b_gamma   = bp.get('focal_gamma', 2.0)
b_mixup   = bp['use_mixup']
b_mixup_a = bp.get('mixup_alpha', 0.0)
b_aux     = bp['use_aux']
b_aux_w   = bp.get('aux_weight', 0.0)
b_swa     = bp['use_swa']
b_contr   = bp['use_contrastive']

cc = None
if b_contr:
    cc = CONTRASTIVE_CKPT
    if not Path(cc).exists():
        print('Contrastive pretraining for final model...')
        contrastive_pretrain(ACTIVE_DS, cc, CONTRASTIVE_EPOCHS, b_lr, b_wd,
                             CONTRASTIVE_TEMP, GRAD_CLIP, BATCH_SIZE)

print(f'Final retrain: fusion={b_fusion} loss={b_loss} swa={b_swa} '
      f'mixup={b_mixup} aux={b_aux} contrastive={b_contr}')
final_f1, history, _ = train_trial(
    fusion=b_fusion, lr=b_lr, dropout=b_dropout, weight_decay=b_wd, loss_type=b_loss,
    label_smoothing=b_ls, focal_gamma=b_gamma, use_mixup=b_mixup, mixup_alpha=b_mixup_a,
    use_aux=b_aux, aux_weight=b_aux_w, use_swa=b_swa,
    ds=ACTIVE_DS, total_epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE,
    contrastive_ckpt=cc, ckpt_out=BEST_CKPT, trial=None, verbose=True)
print(f'\nFinal model saved -> {BEST_CKPT} | val f1_macro {final_f1:.4f}')

## 5 — Evaluation + deliverables

Load the final checkpoint, evaluate on the held-out test set, and write the per-dataset
deliverables: 7-metric table (CSV+JSON), confusion matrix PNG, per-class report, and
training curves PNG.

**Test evaluation.** Loads the final checkpoint, evaluates on the held-out test set, prints and saves the 7-metric table (CSV + JSON).

In [ ]:
ckpt = torch.load(BEST_CKPT, map_location=device, weights_only=False)
final_model = ImprovedDualBranchFusion(
    num_classes=len(class_names), dropout=ckpt['dropout'],
    fusion=ckpt['fusion'], pretrained=False, use_aux=ckpt['use_aux']).to(device)
final_model.load_state_dict(ckpt['model_state_dict'])
final_model.eval()

_, _, test_loader, _ = make_loaders(BATCH_SIZE, ACTIVE_DS)
metrics, y_true, y_pred, y_prob = evaluate_model_full(final_model, test_loader)

results = {'dataset': DATASET_NAME, 'config': ckpt['config'], 'metrics': metrics}
with open(RESULTS_JSON, 'w') as f:
    json.dump(results, f, indent=2)
df_res = pd.DataFrame([metrics], index=[f'thesis_final_{DATASET_NAME}'])
df_res.index.name = 'Model'
df_res.to_csv(RESULTS_CSV)
print(f'Saved {RESULTS_JSON} + {RESULTS_CSV}\n')
print('Test-set metrics:')
for k, v in metrics.items():
    print(f'  {k:<20}: {v:.4f}')
df_res

**Confusion matrix + per-class report.** Row-normalized confusion matrix (PNG) and the per-class precision/recall/F1 table.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))), normalize='true')
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues', vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names, linewidths=0.5)
plt.title(f'thesis_final · {DATASET_NAME} — Confusion Matrix\n'
          f"(acc={metrics['accuracy']:.3f}, F1-macro={metrics['f1_macro']:.3f})",
          fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout(); plt.savefig(CONF_PNG, dpi=200, bbox_inches='tight'); plt.show()
print(f'Saved {CONF_PNG}\n')
print('Per-class report:')
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

**Training curves.** Loss, accuracy, and val F1-macro per epoch for the final model (PNG).

In [ ]:
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(ep, history['train_loss'], label='train'); axes[0].plot(ep, history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, history['train_acc'], label='train'); axes[1].plot(ep, history['val_acc'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(ep, history['val_f1'], color='#2ca02c'); axes[2].set_title('Val F1-macro')
axes[2].set_xlabel('epoch'); axes[2].grid(alpha=0.3)
plt.suptitle(f'thesis_final · {DATASET_NAME} — final-model training curves', fontweight='bold')
plt.tight_layout(); plt.savefig(CURVES_PNG, dpi=200, bbox_inches='tight'); plt.show()
print(f'Saved {CURVES_PNG}')

## 6 — Cross-machine comparison

Once both machines finish (one per dataset), gather the per-dataset artifacts to compare:

```python
import json, pandas as pd
rows = []
for ds in ('derm7pt', 'milk10k'):
    with open(f'results_{ds}.json') as f:
        r = json.load(f)
    row = {'dataset': ds, **r['metrics'], **{f'cfg_{k}': v for k, v in r['config'].items()}}
    rows.append(row)
pd.DataFrame(rows).set_index('dataset')
```

Each machine also wrote `thesis_final_<ds>_bestparams.json` (the winning Optuna config),
`optuna_<ds>.db` (full study, resumable / inspectable), `confusion_<ds>.png`, and
`curves_<ds>.png`.